## 0. Import

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from math import sqrt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, Permute, Reshape, Dense,
                                     Embedding, GRU, Bidirectional, AdditiveAttention, Concatenate,
                                     BatchNormalization, Dropout)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [ ]:
IMG_SHAPE = (28, 224, 1)

PAD_TOKEN = 10
START_TOKEN = 11
END_TOKEN = 12

VOCAB_SIZE = 13  # 0-9, PAD, START, END

VERSION = "v1"  # ĐỔI TÊN VERSION Ở ĐÂY VỚI MỖI KIẾN TRÚC KHÁC NHAU

## 1. Read dataset

In [ ]:
# Load dữ liệu
images = np.load("../datasets/images.npy", mmap_mode="r")
label_strings = np.load("../datasets/labels.npy")

In [ ]:

# Chuyển label từ chuỗi thành danh sách các chữ số
labels = [[int(digit) for digit in label] for label in label_strings]

# Độ dài nhãn lớn nhất
max_len = max(len(label) for label in labels)

print("Number of images:", len(images))
print("Image shape:", images.shape)
print("Maximum label length:", max_len)

## 2. Preprocessing

In [ ]:
# Padding nhãn cho Encoder:
# Mỗi nhãn cần thêm 1 vị trí dành cho END
target_len = max_len + 1

# Khởi tạo toàn bộ bằng PAD
y = np.full((len(labels), target_len), PAD_TOKEN, dtype='int32')
for i, label in enumerate(labels):
    y[i, :len(label)] = label
    y[i, len(label)] = END_TOKEN

In [ ]:
# Tạo đầu vào cho Decoder:
# Dịch nhãn sang phải 1 bước và chèn START_TOKEN vào đầu
start_col = np.full((len(y), 1), START_TOKEN, dtype='int32')
dec_in = np.concatenate([start_col, y[:, :-1]], axis=1)

In [ ]:
print("Maximum digit length:", max_len)
print("Target length:", target_len)

print("\nExample labels:")
for i in range(3):
    print("Original:", label_strings[i])
    print("Decoder input:", dec_in[i])
    print("Decoder target:", y[i])
    print()

## 3. Data Split

Chia train/val/test = 70/15/15

In [ ]:
idx = np.arange(len(images))
idx_train, idx_tmp = train_test_split(idx, test_size=0.30, random_state=42)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.50, random_state=42)

X_train, X_val, X_test = images[idx_train], images[idx_val], images[idx_test]
D_train, D_val, D_test = dec_in[idx_train], dec_in[idx_val], dec_in[idx_test]
y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]

print('Train:', X_train.shape)
print('Validation:', X_val.shape)
print('Test:', X_test.shape)

## 4. Train model

In [ ]:
units = 128

# Encoder (CNN + Bidirectional GRU)
img_input = Input(shape=IMG_SHAPE)                    # (28, 224, 1)
x = Conv2D(32, (3, 3), padding='same', activation='relu')(img_input)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)                         # (14, 112, 32)
x = Conv2D(64, (3, 3), padding='same', activation='relu')(x)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)                         # (7, 56, 64)
x = Conv2D(128, (3, 3), padding='same', activation='relu')(x)
x = BatchNormalization()(x)
x = MaxPooling2D((7, 1))(x)                         # (1, 56, 128)

# Chuyển feature map thành chuỗi từ trái sang phải, giữ 56 vị trí ngang.
h, w, c = x.shape[1], x.shape[2], x.shape[3]
x = Permute((2, 1, 3))(x)
x = Reshape((w, h * c))(x)                          # (56, 128)

# Mỗi chiều GRU có 64 units; ghép lại thành 128 đặc trưng.
enc_out, forward_h, backward_h = Bidirectional(
    GRU(units // 2, return_sequences=True, return_state=True),
    merge_mode='concat'
)(x)
state_h = Concatenate()([forward_h, backward_h])    # (batch, 128)

# Decoder (GRU + Additive Attention)
dec_input = Input(shape=(target_len,))
emb = Embedding(VOCAB_SIZE, 64)(dec_input)
dec_out = GRU(units, return_sequences=True)(emb, initial_state=[state_h])

context = AdditiveAttention()([dec_out, enc_out])
merged = Concatenate()([dec_out, context])
z = Dense(256, activation='relu')(merged)
outputs = Dense(VOCAB_SIZE, activation='softmax')(z)


In [ ]:
model = Model(inputs=[img_input, dec_input], outputs=outputs)
model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', weighted_metrics=['sparse_categorical_accuracy'])
model.summary()

In [ ]:
# Mask những vị trí PAD
train_mask = (y_train != PAD_TOKEN).astype('float32')
val_mask = (y_val != PAD_TOKEN).astype('float32')

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

# Training
history = model.fit([X_train, D_train], y_train, 
                    sample_weight=train_mask, 
                    epochs=50, 
                    batch_size=64, 
                    verbose=1, 
                    validation_data=([X_val, D_val], y_val, val_mask), 
                    callbacks=[early_stop, reduce_lr])

## 5. Prediction

In [ ]:
# khi dự đoán không có nhãn thật nên phải sinh từng chữ số một,
# chữ số vừa sinh ra được đưa vào làm input cho bước kế tiếp

# Khởi tạo Decoder input với toàn bộ là PAD_TOKEN, riêng vị trí đầu tiên là START_TOKEN
decoder_input = np.full((len(X_test), target_len), PAD_TOKEN, dtype='int32')
decoder_input[:, 0] = START_TOKEN

# Mảng lưu kết quả dự đoán và trạng thái hoàn thành của từng câu
y_pred = np.full((len(X_test), target_len), PAD_TOKEN, dtype='int32')
is_finished = np.zeros(len(X_test), dtype=bool)

# Dự đoán từng bước một (Greedy Decoding)
for t in range(target_len):
    probs = model.predict([X_test, decoder_input], batch_size=256, verbose=0)
    next_token = np.argmax(probs[:, t, :], axis=-1)

    # Nếu câu đã kết thúc ở các bước trước thì chỉ điền PAD_TOKEN
    next_token[is_finished] = PAD_TOKEN
    y_pred[:, t] = next_token

    # Cập nhật trạng thái cho những câu vừa sinh ra END_TOKEN
    is_finished = is_finished | (next_token == END_TOKEN)
    
    # Đưa token vừa dự đoán vào làm input cho bước tiếp theo (nếu câu chưa kết thúc)
    if t + 1 < target_len:
        not_finished = (is_finished == False)
        decoder_input[not_finished, t + 1] = next_token[not_finished]

## 6. Evaluation

In [ ]:
# Chuyển chuỗi token thành chuỗi chữ số
def decode_sequence(seq):
    digits = []
    for token in seq:
        if token >= 10:
            break
        digits.append(str(token))
    return ''.join(digits)


# Chuỗi thực tế
true_sequences = [''.join(map(str, labels[i])) for i in idx_test]

# Chuỗi dự đoán
pred_sequences = [decode_sequence(seq) for seq in y_pred]

In [ ]:
# SEQUENCE ACCURACY
correct = np.array([true == pred for true, pred in zip(true_sequences, pred_sequences)])
seq_acc = np.mean(correct)
print(f"Sequence Accuracy: {seq_acc:.4f}")

In [ ]:
# NUMERIC METRICS
# Chỉ tính trên các mẫu mà cả actual và prediction đều là chuỗi hợp lệ
valid = np.array([
    true != '' and pred != ''
    for true, pred in zip(true_sequences, pred_sequences)
])

true_num = np.array([int(true) for true in np.array(true_sequences)[valid]], dtype=np.float64)
pred_num = np.array([int(pred) for pred in np.array(pred_sequences)[valid]], dtype=np.float64)

mse = mean_squared_error(true_num, pred_num)
mae = mean_absolute_error(true_num, pred_num)
r2 = r2_score(true_num, pred_num)
# NSE
nse_denominator = np.sum((true_num - np.mean(true_num)) ** 2)
nse = 1 - np.sum((true_num - pred_num) ** 2) / nse_denominator if nse_denominator != 0 else np.nan

print(f"MAE:  {mae:.4f}")
print(f"RMSE: {sqrt(mse):.4f}")
print(f"R2:   {r2:.4f}")
print(f"NSE:  {nse:.4f}")

- bước preprocess ảnh nên cho vào một file scripts.py để đỡ phải chạy lại, lưu vào folder datasets/ luôn
- thống nhất toàn bộ flow code từ đầu đến cuối, chỉ khác cấu hình và kiến trúc model
- viết lại Base_Model
- model thì lưu vào models/
- mỗi notebook ở cuối file có đoạn code riêng để lưu kết quả từng model thử nghiệm ra csv vào folder output/

### Save Result

In [ ]:
output_path = "../output/KietModel/"  
csv_path = output_path + "kiet_model_results.csv"   
model_path = output_path + f"kiet_model_{VERSION}.keras"   

import os
os.makedirs(output_path, exist_ok=True)

model.save(model_path)  

# Collect evaluation results
results = {
    "Model": f"Kiet Model {VERSION}",     
    "Version": VERSION,
    "Sequence_Accuracy": seq_acc,
    "MAE": mae,
    "RMSE": sqrt(mse),
    "R2": r2,
    "NSE": nse
}

results_df = pd.DataFrame([results])

if os.path.exists(csv_path):
    results_df = pd.concat([pd.read_csv(csv_path), results_df], ignore_index=True)
results_df.to_csv(csv_path, index=False)

print(f"Model saved: {model_path}")
print(f"Results saved: {csv_path}")